## 0. Colab Environment Setup
Mount Google Drive, clone the repo, install deps. **Skip this section if running locally.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Mount Google Drive (Colab only — skip locally)
# ═══════════════════════════════════════════════════════════════
import pathlib

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping Drive mount")

if IN_COLAB:
    DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/phase_lm")
    DRIVE_DATA = DRIVE_ROOT / "data"
    DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
    DRIVE_PROC = DRIVE_ROOT / "processed_data"

    for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROC]:
        d.mkdir(parents=True, exist_ok=True)

    LOCAL_ROOT = pathlib.Path("/content/phase_lm")

    print("Google Drive mounted.")
    print(f"  Drive checkpoints : {DRIVE_CKPT}")
    print(f"  Drive data        : {DRIVE_DATA}")

    csv_files = list(DRIVE_DATA.glob("*.csv"))
    print(f"\nCSV files on Drive: {len(csv_files)}")
else:
    LOCAL_ROOT = pathlib.Path.cwd()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Clone repo + symlink data (Colab only)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

if IN_COLAB:
    GITHUB_URL = "https://github.com/Unseengap/wavetrader.git"
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    def _run(cmd: str) -> None:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.stdout: print(result.stdout)
        if result.stderr: print(result.stderr)

    if not (LOCAL_ROOT / ".git").exists():
        print("Cloning from GitHub …")
        _run(f"git clone {GITHUB_URL} {LOCAL_ROOT}")
    else:
        print("Repo already cloned — pulling latest …")
        _run(f"cd {LOCAL_ROOT} && git stash && git pull origin main")

    # Add repo to Python path
    sys.path.insert(0, str(LOCAL_ROOT))
    print(f"Repo root: {LOCAL_ROOT}")

    # Symlink data/ → Drive (avoids duplicating ~30 GB)
    local_data_link = LOCAL_ROOT / "data"
    if not local_data_link.exists():
        local_data_link.symlink_to(DRIVE_DATA)
        print(f"Symlinked {local_data_link} → {DRIVE_DATA}")

    # Symlink processed_data/ → Drive
    local_proc_link = LOCAL_ROOT / "processed_data"
    if not local_proc_link.exists():
        local_proc_link.symlink_to(DRIVE_PROC)
        print(f"Symlinked {local_proc_link} → {DRIVE_PROC}")

    # Change working dir so relative paths resolve
    import os
    os.chdir(str(LOCAL_ROOT))
    print(f"Working dir: {os.getcwd()}")
else:
    print("Local environment — repo already available")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Install dependencies (Colab only — already in local venv)
# ═══════════════════════════════════════════════════════════════
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "pyarrow", "fastparquet", "plotly"],
        check=True,
    )
    print("Dependencies installed.")
else:
    print("Local environment — skipping pip install")

# MeanReversion Training Notebook
## Pair-Agnostic Multi-Timeframe Mean-Reversion — GPU Training

**MeanReversion** profits when prices snap back to their average after overextension.
It detects Bollinger Band / z-score extremes and trades the reversion, targeting ~71% win rates
with tighter TP (back to mean) and wider SL (allow overshoot).

**Data**: Reuses the same processed parquet data from the MTF pipeline.

**OANDA Account**: `TBD` (separate from MTF and WaveFollower)

In [ ]:
import sys, os, time, json, hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.amp import autocast, GradScaler

# Ensure wavetrader is importable (Colab path set above; local fallback here)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    # Performance flags — ALL mandatory
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"cuDNN benchmark : enabled")
    print(f"TF32            : enabled")
else:
    print("WARNING: No GPU detected — training will use CPU (slow).")

## Configuration & Data Paths

In [ ]:
from wavetrader.mean_reversion import MeanReversion
from wavetrader.config import MeanRevConfig

config = MeanRevConfig(
    learning_rate=2e-4,   # Scale with batch size: lr *= batch/128
    batch_size=320,       # Reduced from 640 — more gradient noise helps generalization
    epochs=30,            # Reduced from 60 — model converges early
    dropout=0.45,         # Increased from 0.30 — stronger regularization against overfitting
    label_smoothing=0.1,  # Prevents overconfident logits
)

PROCESSED_DIR = Path("processed_data")
CHECKPOINT_DIR = Path("checkpoints/mean_reversion")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["GBPJPY", "EURJPY", "USDJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY",
              "USDJPY": "USD/JPY", "GBPUSD": "GBP/USD"}

print(f"Config: {config}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

# ── Check which pairs have processed data — skip missing ones ────────
def _pair_has_data(processed_dir, pair, timeframes):
    for split in ["train", "val", "test"]:
        for tf in timeframes:
            tf_short = tf.replace("min", "m")
            if not (processed_dir / split / f"{pair}_{tf_short}.parquet").exists():
                return False
    return True

available_pairs = [p for p in PAIRS if _pair_has_data(PROCESSED_DIR, p, config.timeframes)]
skipped_pairs = [p for p in PAIRS if p not in available_pairs]

if skipped_pairs:
    print(f"⚠️  Skipping pairs with missing data: {skipped_pairs}")
print(f"Training pairs: {available_pairs}")

PAIRS = available_pairs
assert len(PAIRS) > 0, "No pairs have processed data — run preprocessing first!"

## Load Data from Processed Parquets

In [ ]:
from wavetrader.data import _normalise_df

def load_split_data(split, pairs, timeframes, processed_dir):
    all_pair_data = {}
    for pair_tag in pairs:
        dfs = {}
        for tf in timeframes:
            tf_short = tf.replace("min", "m")
            path = processed_dir / split / f"{pair_tag}_{tf_short}.parquet"
            df = pd.read_parquet(path)
            df = _normalise_df(df)
            dfs[tf] = df
        all_pair_data[pair_tag] = dfs
    return all_pair_data

print("Loading train split...")
train_data = load_split_data("train", PAIRS, config.timeframes, PROCESSED_DIR)
print("Loading val split...")
val_data = load_split_data("val", PAIRS, config.timeframes, PROCESSED_DIR)
print("Loading test split...")
test_data = load_split_data("test", PAIRS, config.timeframes, PROCESSED_DIR)

for pair in PAIRS:
    sizes = {tf: len(df) for tf, df in train_data[pair].items()}
    print(f"  {pair} train: {sizes}")

## Verify Data Format Compatibility

In [ ]:
REQUIRED_FEATURES = {
    "ohlcv":     ["open_norm", "high_norm", "low_norm", "close_norm", "volume_norm"],
    "structure": [f"structure_{i}" for i in range(8)],
    "rsi":       ["rsi_norm", "rsi_delta_norm", "rsi_accel_norm"],
    "volume":    ["volume_norm", "volume_ratio", "volume_delta"],
}

sample_pair = PAIRS[0]
sample_tf = config.timeframes[0]
sample_df = train_data[sample_pair][sample_tf]
print(f"Sample: {sample_pair} {sample_tf} — {len(sample_df)} rows, {len(sample_df.columns)} cols")
print(f"Columns: {list(sample_df.columns[:20])}...")

for group, cols in REQUIRED_FEATURES.items():
    missing = [c for c in cols if c not in sample_df.columns]
    status = "OK" if not missing else f"MISSING: {missing}"
    print(f"  {group:12s}: {status}")

## Build PreCachedMTFDataset + DataLoaders

In [ ]:
from wavetrader.dataset import MTFForexDataset, mtf_collate_fn
from torch.utils.data import ConcatDataset, Dataset, DataLoader as _DL

class PreCachedMTFDataset(Dataset):
    """
    Wraps MTFForexDataset and pre-caches ALL samples as contiguous tensors.
    Uses a multi-worker DataLoader internally to parallelize the caching —
    ~10-15x faster than sample-by-sample on Colab's weak CPUs.
    """

    def __init__(self, mtf_dataset: MTFForexDataset):
        n = len(mtf_dataset)
        print(f"    Pre-caching {n:,} samples...", end=" ", flush=True)
        t0 = time.time()

        # Probe structure from first sample
        sample = mtf_dataset[0]
        self.timeframes = [k for k in sample if k not in ("label", "trend_label", "add_target")]
        self.feat_keys = {tf: list(sample[tf].keys()) for tf in self.timeframes}

        # Pre-allocate contiguous tensors
        self.tensors = {}
        for tf in self.timeframes:
            self.tensors[tf] = {}
            for feat in self.feat_keys[tf]:
                shape = sample[tf][feat].shape
                self.tensors[tf][feat] = torch.empty(n, *shape, dtype=torch.float32)
        self.labels = torch.empty(n, dtype=torch.long)

        # Batch-load with workers instead of for i in range(n)
        cache_workers = min(4, os.cpu_count() or 1)
        cache_bs = 512
        loader = _DL(
            mtf_dataset,
            batch_size=cache_bs,
            shuffle=False,
            num_workers=cache_workers,
            collate_fn=mtf_collate_fn,
            pin_memory=False,
            persistent_workers=False,
            prefetch_factor=2 if cache_workers > 0 else None,
        )

        idx = 0
        for batch in loader:
            bs = batch["label"].shape[0]
            self.labels[idx:idx+bs] = batch["label"]
            for tf in self.timeframes:
                for feat in self.feat_keys[tf]:
                    self.tensors[tf][feat][idx:idx+bs] = batch[tf][feat]
            idx += bs

        elapsed = time.time() - t0
        rate = n / max(elapsed, 0.1)
        print(f"done in {elapsed:.0f}s ({rate:,.0f} samples/s)")

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        result = {}
        for tf in self.timeframes:
            result[tf] = {feat: self.tensors[tf][feat][idx] for feat in self.feat_keys[tf]}
        result["label"] = self.labels[idx]
        return result


# Build per-pair datasets then concatenate
print("Building training datasets...")
train_datasets = []
for pair_tag in PAIRS:
    ds = MTFForexDataset(train_data[pair_tag], config)
    cached = PreCachedMTFDataset(ds)
    train_datasets.append(cached)
    print(f"  {pair_tag}: {len(cached):,} samples")

train_dataset = ConcatDataset(train_datasets)
print(f"Total training samples: {len(train_dataset):,}")

print("\nBuilding validation datasets...")
val_datasets = []
for pair_tag in PAIRS:
    ds = MTFForexDataset(val_data[pair_tag], config)
    cached = PreCachedMTFDataset(ds)
    val_datasets.append(cached)
    print(f"  {pair_tag}: {len(cached):,} samples")

val_dataset = ConcatDataset(val_datasets)
print(f"Total validation samples: {len(val_dataset):,}")

# DataLoaders — GPU-optimized settings
NUM_WORKERS = min(4, os.cpu_count() or 1)
PREFETCH = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

print(f"\nTrain batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}")
print(f"Workers: {NUM_WORKERS}  |  Prefetch: {PREFETCH}  |  Pin memory: {device.type == 'cuda'}")

## Instantiate Model + torch.compile + Sanity Check

In [ ]:
model = MeanReversion(config).to(device)
print(f"MeanReversion: {model.count_parameters():,} trainable parameters")

# torch.compile — fuses kernels, eliminates Python overhead (15-30% speedup)
if device.type == "cuda" and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("torch.compile: enabled (reduce-overhead mode)")
    except Exception as e:
        print(f"torch.compile: skipped ({e})")

# Sanity check — also triggers compile warmup
sample_batch = next(iter(train_loader))
model_input = {
    tf: {k: v.to(device) for k, v in sample_batch[tf].items()}
    for tf in config.timeframes
    if tf in sample_batch and isinstance(sample_batch[tf], dict)
}
with torch.no_grad():
    out = model(model_input)
print("Forward pass OK — output shapes:",
      {k: list(v.shape) for k, v in out.items() if isinstance(v, torch.Tensor)})

## Loss, Optimizer, Scheduler, AMP Setup

In [ ]:
from wavetrader.train_mean_reversion import MeanRevLoss

criterion = MeanRevLoss(label_smoothing=config.label_smoothing).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=0.08,
    betas=(0.9, 0.999),
    fused=(device.type == "cuda"),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.epochs, eta_min=config.learning_rate * 0.01,
)

use_amp = device.type == "cuda"
scaler = GradScaler("cuda", enabled=use_amp)
print(f"AMP: {'enabled' if use_amp else 'disabled'}")
print(f"Optimizer: AdamW (fused={device.type == 'cuda'}, weight_decay=0.08)")
print(f"Scheduler: CosineAnnealing (T_max={config.epochs})")
print(f"Label smoothing: {config.label_smoothing}")

## Training Loop (Inline — with AMP, per-epoch validation, best checkpoint saving)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, config, use_amp):
    model.train()
    total_loss, correct, n_samples = 0.0, 0, 0
    component_sums = {"signal_loss": 0.0, "conf_loss": 0.0, "ext_loss": 0.0, "regime_loss": 0.0}

    for batch in loader:
        labels = batch["label"].to(device, non_blocking=True)
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, labels)

        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)  # MUST come before clip_grad_norm_
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses["total"].item()
        for k in component_sums:
            if k in losses:
                component_sums[k] += losses[k].item()
        correct += (out["signal_logits"].argmax(-1) == labels).sum().item()
        n_samples += labels.size(0)

    n_batches = max(len(loader), 1)
    return {
        "loss": total_loss / n_batches,
        "accuracy": correct / max(n_samples, 1),
        **{k: v / n_batches for k, v in component_sums.items()},
    }


@torch.no_grad()
def validate(model, loader, criterion, device, config, use_amp):
    model.eval()
    total_loss, correct, n_samples = 0.0, 0, 0

    for batch in loader:
        labels = batch["label"].to(device, non_blocking=True)
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, labels)
        total_loss += losses["total"].item()
        correct += (out["signal_logits"].argmax(-1) == labels).sum().item()
        n_samples += labels.size(0)

    n_batches = max(len(loader), 1)
    return {"loss": total_loss / n_batches, "accuracy": correct / max(n_samples, 1)}


# ── Main training loop ──────────────────────────────────────────────────
best_val_acc = 0.0
PATIENCE = 10       # Early stopping — stop after 10 epochs with no val improvement
no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = CHECKPOINT_DIR / f"mean_reversion_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)
best_path = run_dir / "model_weights.pt"

print(f"Training up to {config.epochs} epochs (early stop patience={PATIENCE}) — saving best to {run_dir}")
print(f"{'Epoch':>5} {'Train Loss':>10} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>10} {'LR':>10} {'Best':>5}")
print("-" * 65)

t0 = time.time()
for epoch in range(config.epochs):
    train_m = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, config, use_amp)
    val_m = validate(model, val_loader, criterion, device, config, use_amp)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_m["loss"])
    history["val_loss"].append(val_m["loss"])
    history["train_acc"].append(train_m["accuracy"])
    history["val_acc"].append(val_m["accuracy"])
    history["lr"].append(lr)

    is_best = val_m["accuracy"] > best_val_acc
    if is_best:
        best_val_acc = val_m["accuracy"]
        no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_accuracy": best_val_acc,
            "config": config.__dict__,
        }, str(best_path))
    else:
        no_improve += 1

    print(f"{epoch+1:5d} {train_m['loss']:10.4f} {val_m['loss']:10.4f} "
          f"{train_m['accuracy']:10.4f} {val_m['accuracy']:10.4f} {lr:10.2e} "
          f"{'  *' if is_best else f'  ({no_improve}/{PATIENCE})'}")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} — no val improvement for {PATIENCE} epochs")
        break

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/60:.1f} min — best val acc: {best_val_acc:.4f} — ran {epoch+1}/{config.epochs} epochs")

## Training Curves

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=["Loss", "Accuracy", "Learning Rate", "Overfit Gap"])

    epochs_x = list(range(1, len(history["train_loss"]) + 1))

    fig.add_trace(go.Scatter(x=epochs_x, y=history["train_loss"], name="Train Loss",
                             line=dict(color="#FF6B6B")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs_x, y=history["val_loss"], name="Val Loss",
                             line=dict(color="#4ECDC4")), row=1, col=1)

    fig.add_trace(go.Scatter(x=epochs_x, y=history["train_acc"], name="Train Acc",
                             line=dict(color="#FF6B6B")), row=1, col=2)
    fig.add_trace(go.Scatter(x=epochs_x, y=history["val_acc"], name="Val Acc",
                             line=dict(color="#4ECDC4")), row=1, col=2)

    fig.add_trace(go.Scatter(x=epochs_x, y=history["lr"], name="LR",
                             line=dict(color="#FFE66D")), row=2, col=1)

    overfit = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
    fig.add_trace(go.Scatter(x=epochs_x, y=overfit, name="Overfit Gap",
                             line=dict(color="#FF6B6B"), fill="tozeroy"), row=2, col=2)

    fig.update_layout(template="plotly_dark", height=600, showlegend=True,
                      title="MeanReversion Training Curves")
    fig.show()
except ImportError:
    print("plotly not installed — skipping training curves")

## Test Set Evaluation

In [ ]:
# Reload best checkpoint
best_ckpt = torch.load(str(best_path), map_location=device, weights_only=False)
if hasattr(model, "_orig_mod"):
    model._orig_mod.load_state_dict(
        {k.replace("_orig_mod.", ""): v for k, v in best_ckpt["model_state_dict"].items()}
    )
else:
    model.load_state_dict(
        {k.replace("_orig_mod.", ""): v for k, v in best_ckpt["model_state_dict"].items()}
    )

# Build test datasets
print("Building test datasets...")
test_datasets = []
for pair_tag in PAIRS:
    ds = MTFForexDataset(test_data[pair_tag], config)
    cached = PreCachedMTFDataset(ds)
    test_datasets.append(cached)
    print(f"  {pair_tag}: {len(cached):,} samples")

test_dataset = ConcatDataset(test_datasets)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
)

test_m = validate(model, test_loader, criterion, device, config, use_amp)
print(f"\nTest Loss: {test_m['loss']:.4f}  |  Test Accuracy: {test_m['accuracy']:.4f}")

# Per-class accuracy
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        labels = batch["label"].to(device, non_blocking=True)
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
        all_preds.append(out["signal_logits"].argmax(-1).cpu())
        all_labels.append(labels.cpu())

preds = torch.cat(all_preds)
labels = torch.cat(all_labels)
class_names = ["BUY", "SELL", "HOLD"]
for i, name in enumerate(class_names):
    mask = labels == i
    if mask.sum() > 0:
        acc = (preds[mask] == i).float().mean().item()
        print(f"  {name:5s}: {acc:.4f} ({mask.sum().item()} samples)")

## Save Checkpoint + SHA-256 + History + Google Drive Copy

In [ ]:
# Save config
config_path = run_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config.__dict__, f, indent=2, default=str)
print(f"Config saved to {config_path}")

# Save history
history_path = run_dir / "history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")

# SHA-256 of weights
sha_path = run_dir / "weights.sha256"
h = hashlib.sha256()
with open(best_path, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        h.update(chunk)
sha_hex = h.hexdigest()
sha_path.write_text(f"{sha_hex}  model_weights.pt\n")
print(f"SHA-256: {sha_hex}")

# Copy to Google Drive
try:
    import shutil
    if IN_COLAB:
        drive_dir = DRIVE_CKPT / run_dir.name
    else:
        drive_dir = Path("/content/drive/MyDrive/phase_lm/checkpoints") / run_dir.name
    if drive_dir.parent.exists():
        shutil.copytree(str(run_dir), str(drive_dir), dirs_exist_ok=True)
        print(f"Copied to Google Drive: {drive_dir}")
    else:
        print("Google Drive not mounted — skipping copy")
except Exception as e:
    print(f"Drive copy skipped: {e}")

print(f"\nCheckpoint directory: {run_dir}")
print(f"Files: {[f.name for f in run_dir.iterdir()]}")